# 📊 Pair Trading Analysis

**Simple interface for pair trading analysis on Brazilian market (B3)**

1. **Adjust parameters** below if needed
2. **Run all cells** (Ctrl+Shift+P → "Run All") 
3. **View results** at the end

## Setup

In [30]:
# Install dependencies
%pip install -q PyYAML pandas numpy statsmodels tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# Import system (restart to get latest changes)
import sys
from pathlib import Path

# Clear any cached modules
modules_to_clear = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_clear:
    if module in sys.modules:
        del sys.modules[module]

sys.path.insert(0, str(Path().absolute()))
sys.path.insert(0, str(Path().absolute() / 'src'))

import src.pipeline.pair_trading_pipeline as ptp
import src.utils.config_manager as cm
import warnings
warnings.filterwarnings('ignore')

print("✅ System ready! (modules reloaded)")

✅ System ready! (modules reloaded)


In [32]:
# PARAMETERS (modify if needed)
# For data collection
# Data files to use (in data/ folder)
COTAHIST_FILES = [
    "data/COTAHIST_A2014.TXT",
    "data/COTAHIST_A2015.TXT"
    # "data/COTAHIST_A2016.TXT"  # Uncomment to include 2016
]

# Analysis parameters
TOP_N_STOCKS = 100          # Top liquid stocks to select
MIN_VOLUME = 500000         # Minimum daily volume (R$)
P_VALUE_MAX = 0.05          # Maximum p-value for cointegration
HURST_MAX = 0.5             # Maximum Hurst exponent
MIN_CROSSINGS = 24          # Minimum mean crossings in 2 years

print("Parameters set ✅")
print(f"Using {len(COTAHIST_FILES)} COTAHIST files")

Parameters set ✅
Using 2 COTAHIST files


In [33]:
# Apply parameters
config = cm.ConfigManager()

# Set COTAHIST files to use
config.set('data.cotahist_files', COTAHIST_FILES)

# Set analysis parameters
config.set('liquidity.selection.top_n_stocks', TOP_N_STOCKS)
config.set('liquidity.selection.min_daily_volume', MIN_VOLUME)
config.set('analysis.cointegration.significance_level', P_VALUE_MAX)
config.set('analysis.hurst.max_hurst_exponent', HURST_MAX)
config.set('analysis.cross_mean.min_crossings_2y', MIN_CROSSINGS)

print("Parameters applied ✅")
print(f"COTAHIST files: {[f.split('/')[-1] for f in COTAHIST_FILES]}")

Parameters applied ✅
COTAHIST files: ['COTAHIST_A2014.TXT', 'COTAHIST_A2015.TXT']


## Run Analysis

In [ ]:
# Initialize pipeline (reload ALL modules for latest fixes)
import importlib
import sys

# Force reload all src modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

# Also reload the main imports
importlib.reload(ptp)
importlib.reload(cm)

pipeline = ptp.PairTradingPipeline()
print("Pipeline ready 🚀 (ALL modules reloaded)")

# Clear cache to force using new organized folders
pipeline.clear_cache()
print("Cache cleared - will use new folder structure")

# Suppress verbose logging
import logging
logging.getLogger().setLevel(logging.WARNING)

Pipeline ready 🚀
2025-09-19 11:10:00,369 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - Cache do pipeline limpo
Cache cleared - will use new folder structure


In [35]:
# Reload modules to get latest changes
import importlib
importlib.reload(cm)

# Verify output paths
print("📁 Output directories:")
test_config = cm.ConfigManager()
print(f"   Liquidity: {test_config.get_output_path('test.csv', 'liquidity')}")
print(f"   Price matrix: {test_config.get_output_path('test.csv', 'price_matrix')}")
print(f"   Cointegration: {test_config.get_output_path('test.csv', 'cointegration')}")
print(f"   Final results: {test_config.get_output_path('test.csv', 'final_results')}")
print("✅ Organized folders will be used")

📁 Output directories:
   Liquidity: c:\Users\diego\Documents\Itaú\pair-trading\outputs\01_liquidity\test.csv
   Price matrix: c:\Users\diego\Documents\Itaú\pair-trading\outputs\02_price_matrix\test.csv
   Cointegration: c:\Users\diego\Documents\Itaú\pair-trading\outputs\03_cointegration\test.csv
   Final results: c:\Users\diego\Documents\Itaú\pair-trading\outputs\05_final_results\test.csv
✅ Organized folders will be used


In [36]:
# Debug: Check method signature
import inspect
test_config = cm.ConfigManager()
sig = inspect.signature(test_config.get_output_path)
print(f"Method signature: {sig}")

# Try with fallback approach
try:
    # Try new signature
    liquidity_path = test_config.get_output_path('test.csv', 'liquidity')
    print("✅ New signature works")
except TypeError:
    # Fallback to old signature
    print("⚠️ Using fallback - updating paths manually")
    from pathlib import Path
    base_path = Path().absolute()
    
    liquidity_path = str(base_path / 'outputs/01_liquidity/test.csv')
    price_path = str(base_path / 'outputs/02_price_matrix/test.csv')
    coin_path = str(base_path / 'outputs/03_cointegration/test.csv')
    final_path = str(base_path / 'outputs/05_final_results/test.csv')
    
    print("📁 Output directories (manual):")
    print(f"   Liquidity: {liquidity_path}")
    print(f"   Price matrix: {price_path}")
    print(f"   Cointegration: {coin_path}")
    print(f"   Final results: {final_path}")
    print("✅ Organized folders configured")

Method signature: (filename: str, category: str = None) -> str
✅ New signature works


In [38]:
# RUN COMPLETE ANALYSIS
# Usually takes 4 to 5 minutes for 2 years of data
print("🚀 Running analysis...")

results = pipeline.execute_full_pipeline()

stats = pipeline.get_statistics()
print(f"✅ Analysis complete! {stats['final_selected_pairs']} pairs found")

🚀 Running analysis...
2025-09-19 11:17:06,215 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - === INICIANDO PIPELINE COMPLETO DE PAIR TRADING ===
2025-09-19 11:17:06,216 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - === FASE 1: ANÁLISE DE LIQUIDEZ ===
2025-09-19 11:17:06,223 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - Usando resultados em cache da análise de liquidez
2025-09-19 11:17:06,224 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - === FASE 2: PREPARAÇÃO DE DADOS ===
2025-09-19 11:17:06,216 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - === FASE 1: ANÁLISE DE LIQUIDEZ ===
2025-09-19 11:17:06,223 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - Usando resultados em cache da análise de liquidez
2025-09-19 11:17:06,224 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - INFO - === FASE 2: PREPARAÇÃO DE DADOS ===
2025-09-19 11:17:06,225 - src.pipeline.pair_trading_

Fase 2 - Preparação de Dados falhou em 0.00s - Arquivo de tickers não encontrado: top_100_acoes_liquidas_2014_2015_completo.csv


2025-09-19 11:17:06,228 - src.pipeline.pair_trading_pipeline.PairTradingPipeline - ERROR - Erro no pipeline: Arquivo de tickers não encontrado: top_100_acoes_liquidas_2014_2015_completo.csv


FileNotFoundError: Arquivo de tickers não encontrado: top_100_acoes_liquidas_2014_2015_completo.csv

## Results

In [ ]:
# Top liquid stocks
results = pipeline.get_results()
if results['liquidity_analysis'] is not None:
    print("🏆 TOP 10 MOST LIQUID STOCKS:")
    display(results['liquidity_analysis'][['ticker', 'nome_resumido', 'volume_medio_diario']].head(10))

# Cointegrated pairs
if results['cointegrated_pairs'] is not None and not results['cointegrated_pairs'].empty:
    cp = results['cointegrated_pairs']
    print(f"🔬 COINTEGRATED PAIRS: {len(cp)}")
    print(f"Best p-value: {cp['p_valor_coint'].min():.6f}")
    display(cp[['Ativo_A', 'Ativo_B', 'p_valor_coint']].head(5))

In [ ]:
# Final selected pairs
if results['final_pairs'] is not None and not results['final_pairs'].empty:
    final_df = results['final_pairs']
    print(f"🎯 FINAL PAIRS: {len(final_df)}")
    
    cols = ['Ativo_A', 'Ativo_B', 'p_valor_coint']
    if 'hurst_exponent' in final_df.columns:
        cols.append('hurst_exponent')
    
    display(final_df[cols])
    
    best = final_df.iloc[0]
    print(f"💡 BEST PAIR: {best['Ativo_A']}-{best['Ativo_B']} (p-value: {best['p_valor_coint']:.6f})")
else:
    print("❌ No pairs found. Try adjusting parameters.")

In [ ]:
# Summary statistics
stats = pipeline.get_statistics()
print("📊 SUMMARY:")
print(f"   Liquid stocks: {stats['top_liquid_stocks']}")
print(f"   Pairs tested: {stats['total_pair_combinations']:,}")
print(f"   Cointegrated: {stats['cointegrated_pairs']}")
print(f"   Final pairs: {stats['final_selected_pairs']}")

if stats['total_pair_combinations'] > 0:
    rate = stats['final_selected_pairs'] / stats['total_pair_combinations'] * 100
    print(f"   Success rate: {rate:.4f}%")

## Utilities (optional)

In [ ]:
# Clear cache (force recalculation next run)
# pipeline.clear_cache()
# print("Cache cleared ✅")

# Save current configuration
config.save()
print("Config saved ✅")

In [ ]:
# Organize files into folders
import os
import shutil
from pathlib import Path

def organize_all_files():
    """Move existing files to organized folders"""
    root = Path('.')
    
    # Create all directories
    directories = [
        'data',                    # Input data files
        'outputs/01_liquidity',
        'outputs/02_price_matrix',
        'outputs/03_cointegration',
        'outputs/04_filtered_pairs',
        'outputs/05_final_results'
    ]
    
    for dir_path in directories:
        Path(dir_path).mkdir(parents=True, exist_ok=True)
    
    moved_count = 0
    
    # 1. Move COTAHIST files to data/ folder
    cotahist_patterns = ['COTAHIST_A*.TXT', 'COTAHIST_A*.txt']
    for pattern in cotahist_patterns:
        for cotahist_file in root.glob(pattern):
            target_path = Path('data') / cotahist_file.name
            if not target_path.exists():
                shutil.move(str(cotahist_file), str(target_path))
                moved_count += 1
                print(f"   📦 {cotahist_file.name} → data/")
    
    # 2. Move CSV files to appropriate output folders
    csv_dirs = {
        'outputs/01_liquidity': ['*acoes_liquidas*', '*tickers*'],
        'outputs/02_price_matrix': ['matriz_precos*'],
        'outputs/03_cointegration': ['*cointegrados*'],
        'outputs/04_filtered_pairs': ['*filtrados*'],
        'outputs/05_final_results': ['*finais*', '*selecionados*']
    }
    
    for csv_file in root.glob('*.csv'):
        file_moved = False
        for target_dir, patterns in csv_dirs.items():
            for pattern in patterns:
                if csv_file.match(pattern):
                    target_path = Path(target_dir) / csv_file.name
                    if not target_path.exists():
                        shutil.move(str(csv_file), str(target_path))
                        moved_count += 1
                        file_moved = True
                        break
            if file_moved:
                break
    
    # 3. Move TXT files (tickers)
    for txt_file in root.glob('*tickers*.txt'):
        target_path = Path('outputs/01_liquidity') / txt_file.name
        if not target_path.exists():
            shutil.move(str(txt_file), str(target_path))
            moved_count += 1
    
    if moved_count > 0:
        print(f"📁 Organized {moved_count} files into folders")
    else:
        print("📁 All files already organized")
    
    # Show current structure
    print("\n📂 File organization:")
    data_files = list(Path('data').glob('*')) if Path('data').exists() else []
    print(f"   data/: {len(data_files)} files")
    if data_files:
        for f in sorted(data_files):
            print(f"      • {f.name}")
    
    for dir_path in sorted([d for d in directories if d.startswith('outputs')]):
        if Path(dir_path).exists():
            files = list(Path(dir_path).glob('*'))
            print(f"   {dir_path}: {len(files)} files")

# Run organization
organize_all_files()